# 🐝 DataFrames — the real table

**Time:** ~35 minutes &nbsp;•&nbsp; **Goal:** open the pollinator survey **as it actually came off the clipboard**, and learn to interrogate a table you have never seen before.

In the lecture you worked with a tidied extract: 13 neat columns, no missing values, nothing to trip over. That file was made for teaching.

This is the real one. Same survey — Brighton allotments, 2017 and 2018 — but **22 columns**, two years, 83 individual plots, and at least one mistake in the header row.

Learning to open a file like this and work out what is in it is not a preliminary to data analysis. It *is* data analysis, and most of the time it is most of it.

## Our data — nine allotments, two summers

Volunteers walked a plot, picked a flowering crop, watched it for a fixed time, and wrote down every insect that landed. One row = one of those watches.

| column | what it is |
|---|---|
| `Year`, `Date`, `Month` | when the watch happened |
| `Allotment`, `Plot` | where — nine sites, 83 plots between them |
| `Community` | was the plot a shared community plot or an individual one? |
| `Crop_Generic` | which crop was being watched |
| `Plants`, `Total_Flowers` | how many plants, and how many open flowers, were in view |
| `Beetles` … `Wasps` | visit counts, one column per insect group |
| `Terrestris` … `Hortorum` | visit counts for six **bumblebee species** |
| `Bumblebees` | all six species added together |

Run the cell below first. Everything after it depends on `df`.

In [ ]:
# Run this first. It fetches the survey file and opens it.
import urllib.request
from pathlib import Path

import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab06/data/"

SURVEY = "pollinators.csv"
if not Path(SURVEY).exists():
    urllib.request.urlretrieve(DATA_URL + SURVEY, SURVEY)

df = pd.read_csv(SURVEY)
print("loaded", SURVEY)
print(df.shape)

## Skill 1 — A DataFrame will describe itself

You have 1,620 rows. You are never going to read them. So the first thing you do with any new table is ask it four questions:

```python
df.shape      # (rows, columns)
df.columns    # the column names
df.dtypes     # what kind of data is in each column
df.head()     # what do the first few rows actually look like
```

`df.info()` rolls three of those into one, and adds the thing you most need to know: **how many values are actually there**, column by column.

In [ ]:
# DEMO - four questions, asked of a table you have never seen
print("rows and columns:", df.shape)
print()
print(df.columns.tolist())
print()
df.head()

In [ ]:
# DEMO - info() is the one to reach for. Read the 'Non-Null Count' column.
df.info()

### 🐝 DIY challenge

1. Print the **last** three rows with `df.tail(3)`. Which year are they from?
2. `df.shape` is a tuple. Print just the number of **columns**, using an index.
3. Run `df.describe()`. It summarises every *numeric* column. Which columns does it leave out, and why?

> 💡 `.head()` and `.tail()` take a number: `df.head(10)`. `.describe()` needs no argument.

In [ ]:
# TODO - your turn

## Skill 2 — The column that isn't there

Look carefully at the output of `df.info()` above. One of the bumblebee species columns is not called what you think it is called.

This is the single most common way a real analysis fails on day one: somebody typed a space after a column name in Excel three years ago, and now `df["Hortorum"]` raises a `KeyError` while `df["Hortorum "]` works fine — and the two look identical on screen.

The fix is one line, and you should write it into every script you ever start:

```python
df.columns = df.columns.str.strip()
```

`.str` lets you apply a string method to **every name at once**. You will meet `.str` again on data columns; it works the same way.

In [ ]:
# DEMO - the bug, made visible
try:
    df["Hortorum"]
except KeyError as e:
    print("KeyError:", e)

# repr() shows the quotes, so you can finally see the space
print()
for name in df.columns:
    if name != name.strip():
        print("suspicious:", repr(name))

In [ ]:
# DEMO - and the fix
df.columns = df.columns.str.strip()

print(df["Hortorum"].sum(), "Bombus hortorum visits recorded")

### 🐝 DIY challenge

1. Before the fix, how many column names had stray whitespace? Write a line that **counts** them rather than printing them.
2. Column names can be messy in other ways too. Print `df.columns.str.lower()` — what would that be useful for?
3. `Crop_Generic` is a clumsy name. Rename it to `Crop` with
   `df = df.rename(columns={"Crop_Generic": "Crop"})`, then check `df.columns`.

> 💡 A comparison over a whole collection gives you `True`/`False` per item, and `sum()` counts the `True`s — exactly as it did for numpy masks.

In [ ]:
# TODO - your turn

## Skill 3 — A column is a Series

This is the idea that makes pandas click:

> A **DataFrame** is a collection of **Series**, side by side, sharing one index.

So `df["Honeybees"]` is not a list and not an array — it is a Series, a single labelled column. And almost every method you can call on a Series, you can also call on the whole DataFrame, in which case you get **one answer per column**.

```python
df["Honeybees"]          # -> Series
df["Honeybees"].sum()    # -> one number
df[["Honeybees", "Bumblebees"]]   # -> DataFrame (note the double brackets)
```

The double bracket catches everybody. `df["a"]` asks for *one* column and gives you a Series; `df[["a"]]` hands pandas a **list** of names and gives you a DataFrame back.

In [ ]:
# DEMO - one column, and several
bees = df["Honeybees"]

print(type(bees))
print("total honeybee visits:", bees.sum())
print("busiest single watch: ", bees.max())
print("mean per watch:       ", round(bees.mean(), 3))

print()
pair = df[["Honeybees", "Bumblebees"]]
print(type(pair))
pair.head()

### 🐝 DIY challenge

1. How many **bumblebee** visits were recorded in total across the whole survey?
2. Pull out the three columns `Crop_Generic`, `Total_Flowers`, `Bumblebees` and show the first 5 rows.
3. `df["Allotment"]` is a Series of text. Call `.value_counts()` on it. Which allotment was surveyed most?
4. What does `df["Allotment"].nunique()` give you?

> 💡 `.value_counts()` is the single most useful method in pandas for a text column. Use it on everything.

In [ ]:
# TODO - your turn

## Skill 4 — Ask the whole table at once

Because a DataFrame is a stack of Series, a summary method applied to the frame runs down **every column** and hands you back a Series of answers.

```python
df.sum(numeric_only=True)    # one total per numeric column
df.mean(numeric_only=True)   # one mean per numeric column
```

`numeric_only=True` tells pandas to skip the text columns rather than try to add up allotment names.

That one line — `df[insects].sum()` — answers a real question: **which pollinator group is doing the work on Brighton allotments?**

In [ ]:
# DEMO - the headline result of the whole survey, in three lines
insects = ["Beetles", "Hoverflies", "Flies", "Butterflies",
           "Honeybees", "Solitary_Bees", "Wasps", "Bumblebees"]

totals = df[insects].sum()
print(totals.sort_values(ascending=False))

### 🐝 DIY challenge

1. `totals` is itself a Series. What is its `.idxmax()` — and how is that different from its `.max()`?
2. Turn the totals into **percentages** of all insect visits. (Divide by `totals.sum()` and multiply by 100.)
3. Do the same for the six bumblebee species columns. Is one species doing most of the work?

> 💡 `.idxmax()` is the pandas cousin of numpy's `.argmax()`: it returns the **label**, not the position.

In [ ]:
# TODO - your turn

## Skill 5 — Sorting, and the top of the list 🏆

`.sort_values()` reorders the whole table by one column. It does **not** change `df` — it hands you back a sorted copy, which is why you almost always assign it or chain onto it.

```python
df.sort_values("Total_Flowers")                  # smallest first
df.sort_values("Total_Flowers", ascending=False) # largest first
df.sort_values(["Allotment", "Total_Flowers"])   # by one, then the other
```

When you only want the extremes, `.nlargest(n, "column")` is shorter and says what you mean.

In [ ]:
# DEMO - which single watch saw the most bumblebees?
busiest = df.nlargest(5, "Bumblebees")
busiest[["Date", "Allotment", "Crop_Generic", "Total_Flowers", "Bumblebees"]]

### 🏆 DIY finale — a first look at the survey

Answer these four, printing a short line for each:

1. How many watches are in the table, and how many of them recorded **no insects at all**?
   (Build the total with `df[insects].sum(axis=1)` — `axis=1` adds *across* each row rather than down each column.)
2. Which **five** watches saw the most honeybees? Show the date, allotment and crop.
3. Which watch had the most flowers in view? Did it also see the most insects?
4. Sort the whole table by `Allotment` and then by `Bumblebees`, largest first within each allotment, and show the first 10 rows.

> 💡 `axis=0` goes down the columns (the default), `axis=1` goes across the rows. Getting this the wrong way round is a rite of passage.

In [ ]:
# TODO - your turn

## 🔭 Explore further

- The pandas **10 minutes** tour — the official whistle-stop: <https://pandas.pydata.org/docs/user_guide/10min.html>
- The survey these data come from: Nicholls et al. (2023), *Scaling up urban agriculture* — allotment pollinator monitoring in Brighton.
- `df.info()` vs `df.describe()` vs `df.head()` — get into the habit of running all three on every new file.

## Recap

You can now:

- open a CSV and size it up with `.shape`, `.columns`, `.dtypes`, `.info()` and `.describe()`
- spot and fix a whitespace bug in a header with `df.columns.str.strip()`
- tell a **Series** from a **DataFrame**, and remember why `df[["a", "b"]]` needs two brackets
- summarise one column, several columns, or the whole table
- sort and take the top of the list with `.sort_values()` and `.nlargest()`

**Next:** `02_dataframes_selecting` — picking out exactly the rows you want, and asking the table questions.